# 01. Imports

**Tier:** Intermediate. **Why you'd use this:** A hypotenuse helper should live in `math`, not be pasted into every notebook. `import` loads that module once and gives you a name to call.

**After this lesson you can:** bind a module with `import`; pull a name with `from`; explain why a second import does not rerun the file.
About 25 minutes. **Uses:** [Definitions](../05-functions/01-definitions-lesson.ipynb)


## Quick review

1. Why put cleanup after `yield` in `finally` for `@contextmanager`?
2. What does `__exit__` returning true mean?

<details><summary>Answers</summary>

So cleanup runs on error. It suppresses the exception.
</details>


## The idea

`import math` runs the module's top level once, caches the module object in `sys.modules`, and binds the name `math` in your scope. `from math import sqrt` binds `sqrt` directly. A later import of the same module reuses the cache instead of running the file again.

<details><summary>Explain more</summary>

Python searches `sys.path` for the module file. The first hit wins. Packages are directories with a path entry and usually an `__init__.py` story you will meet next.
</details>


## See it work

Import `math` twice and watch the cache.


In [1]:
import math
import sys

# 1. First import runs and caches the module object.
first = math
second = __import__("math")
print(first is second)
print("math" in sys.modules)
print(math.hypot(3, 4))

True
True
5.0


## Check yourself

Answer these before you run the cell.

1. Does a second `import math` rerun the module's top level?
2. What name does `from math import sqrt` bind in your scope?


In [2]:
import math
import sys

before = sys.modules["math"]
import math as again

print(again is before)
from math import sqrt

print(sqrt(9))
print("sqrt" in dir() or callable(sqrt))

True
3.0
True


<details><summary>Answers</summary>

No. It reuses `sys.modules`. The name `sqrt`.
</details>


## Why it works this way

Import is a search, a load, and a bind. The search walks `sys.path` entries until a matching module or package is found. The load executes the module's top-level code and stores the resulting module object in `sys.modules` under its name. The bind step attaches that object, or selected attributes, to names in the importing scope. A second import skips load when the name is already in `sys.modules`. Deep dive: [The import system](../../internals/09-imports/01-import-system-lesson.ipynb).

That cache is why import side effects run once per process. It is also why circular imports can see a half-initialized module: the name is already in `sys.modules` while the first load is still running. Keeping imports at the top and avoiding heavy work at import time reduces those surprises.

`from module import name` still loads the whole module into the cache. It only changes which names appear in your namespace. Cost of a cold import is running the module once. Cost of a warm import is a dict lookup in `sys.modules`.

Evidence: two imports return the same module object identity, and `"math" in sys.modules` is true after the first.

Circular imports fail or half-work because the unfinished module is already cached. Split shared names into a third module, or import inside a function when you must break a cycle. Prefer clear package layouts over clever reload tricks. Reload is for exploration, not for production flow.  Measuring `math is sys.modules["math"]` makes the cache concrete. Treat that identity as the rule, and reload as the exception.


In [3]:
import math
import sys

print(math is sys.modules["math"])
import math as other

print(other is math)
print(math.hypot(3, 4))

True
True
5.0


## When to use it

Use `import module` when you will call several names and want the module prefix. Use `from module import name` for a few clear names. Don't rely on import side effects for work that should run every call. Don't assume a second import refreshes code you changed on disk without reload tooling. Prefer absolute imports in project code so renames stay searchable.  Keep import lines at the top unless a cycle forces a local import.


## Common mistake

Expecting `import module` twice to pick up edits you just made to the file. The cache still holds the old module object.


In [4]:
import math
import sys

cached = sys.modules["math"]
import math

print(math is cached)

True


Restart the kernel, or remove the entry and import again, when you truly need a reload during exploration. Prefer saving and re-running as a program for real checks.


In [5]:
import importlib
import math

math = importlib.reload(math)
print(math.hypot(5, 12))

13.0


## Try it

Use `from math import pi` and compute a circle area. Which name is missing from your scope that `import math` would have given you?


## Recap

- `import` loads once and caches in `sys.modules`.
- `from` still loads the module; it only changes local names.
- A second import reuses the cached module object.

**Next:** [practice](01-imports-practice.ipynb)
